# Offline GEO RAG — Colab Edition with True Multimodal + Page Screenshot Retrieval

This notebook builds an **offline multimodal RAG system** for regional PDFs.

It keeps the original text/table RAG flow, and adds three visual retrieval layers:

1. **Caption-based image indexing**  
   Extract embedded PDF images → describe each image with Qwen2.5-VL → index the caption in FAISS/BM25.

2. **Caption-based page screenshot indexing**  
   Render each full PDF page as an image → describe the page with Qwen2.5-VL → index the page caption in FAISS/BM25.

3. **True visual embedding retrieval**  
   Embed actual pixels from embedded images and rendered page screenshots with SigLIP2 → store visual vectors in a separate FAISS index.

Why both embedded images and page screenshots are useful:

- Embedded images catch photos/screenshots/charts that are stored as raster images inside the PDF.
- Page screenshots catch vector charts, tables, diagrams, and layout that `page.get_images()` can miss.
- Caption retrieval is strong for exact labels, chart titles, visible text, and business language.
- Visual embedding retrieval is stronger when layout, chart shape, screenshots, or diagrams matter.
- The final answer can pass retrieved embedded images or page screenshots directly into Qwen2.5-VL for visual reasoning.

This is still simple enough to learn step by step, but it now matches a more modern multimodal document RAG architecture.


---
## Step 0 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

import os
os.chdir('/content/gdrive/MyDrive')

print("✓ Google Drive mounted")
print("   Working directory:", os.getcwd())

Mounted at /content/gdrive
✓ Google Drive mounted
   Working directory: /content/gdrive/MyDrive


---
## Step 1 — Install packages

> **Note:** If the next install cell changes PyTorch/TorchVision, restart the Colab runtime once before continuing. This avoids partially-loaded `torchvision` import errors.

> **TorchCodec note:** This notebook does not process videos, so `torchcodec` is intentionally removed to avoid FFmpeg/libtorchcodec compatibility errors in Colab.


In [2]:
# Environment setup — run this cell first in a fresh Colab runtime.
#
# Why this cell pins torch + torchvision together:
# Transformers imports torchvision for image utilities. If torch and torchvision
# are upgraded separately, you can get:
#   AttributeError: partially initialized module 'torchvision' has no attribute 'extension'
#
# Why we uninstall torchcodec:
# This notebook works with PDF images, not video/audio decoding. Some Colab images
# may have an incompatible torchcodec already installed. If Transformers discovers
# that broken optional package, imports can fail with:
#   RuntimeError: Could not load libtorchcodec
# Removing torchcodec is safer than installing FFmpeg/TorchCodec for a feature we do not use.
#
# After this cell finishes, restart the Colab runtime once:
#   Runtime → Restart session
# Then continue from the verification/import cells.

!pip uninstall -y -q torchcodec || true

!pip install -q -U --no-cache-dir torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu126

# Pin Transformers to a Qwen2.5-VL/SigLIP2-capable release instead of floating to the newest release.
# Reason: newer releases can add optional video/audio backends that are not needed for this PDF-image RAG notebook.
!pip install -q -U --no-cache-dir transformers==4.51.3 accelerate bitsandbytes qwen-vl-utils pillow tqdm numpy pandas scikit-learn faiss-cpu rank_bm25 rouge-score sentence-transformers langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters pymupdf pymupdf4llm

# Remove it again in case any base/runtime package or dependency brought it back.
!pip uninstall -y -q torchcodec || true

print("✓ Packages installed")
print("✓ torchcodec removed because this notebook does image/PDF RAG, not video/audio decoding")
print("IMPORTANT: Restart the Colab runtime once before running the import cells.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 186.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 237.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 262.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 211.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 160.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 204.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 330.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 281.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 302.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 209.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 175.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 244.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Verify the deep-learning stack after restarting the runtime.
# This catches torch/torchvision mismatches and broken optional media packages early.

import importlib.util
import torch
import torchvision
import transformers

print("✓ torch:", torch.__version__)
print("✓ torchvision:", torchvision.__version__)
print("✓ transformers:", transformers.__version__)
print("✓ CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✓ GPU:", torch.cuda.get_device_name(0))
    print("✓ CUDA runtime:", torch.version.cuda)

# torchcodec is optional and not needed here. It should be absent to avoid FFmpeg/libtorchcodec issues.
if importlib.util.find_spec("torchcodec") is None:
    print("✓ torchcodec not installed — good for this image-only notebook")
else:
    print("⚠ torchcodec is installed. If you see libtorchcodec/FFmpeg errors, run: !pip uninstall -y torchcodec, then restart runtime.")


✓ torch: 2.7.0+cu126
✓ torchvision: 0.22.0+cu126
✓ transformers: 4.51.3
✓ CUDA available: True
✓ GPU: Tesla T4
✓ CUDA runtime: 12.6
✓ torchcodec not installed — good for this image-only notebook


In [1]:
# Emergency fix only — run this if you still see `Could not load libtorchcodec`.
# Then restart the runtime and continue from the verification cell.

!pip uninstall -y torchcodec
print("✓ torchcodec removed. Now restart the runtime before continuing.")


✓ torchcodec removed. Now restart the runtime before continuing.


In [ ]:
# Optional: unzip /content/Archive.zip into the persistent Drive data folder.
# Reason: this cell used to depend on `settings` before settings existed, which can break a fresh runtime.
import zipfile
from pathlib import Path

archive_path = Path('/content/Archive.zip')
extraction_path = Path('/content/gdrive/MyDrive/RAG_data/data')
extraction_path.mkdir(parents=True, exist_ok=True)

if archive_path.exists():
    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"✓ Archive extracted to: {extraction_path}")
else:
    print(f"ℹ No archive found at {archive_path}. Upload PDFs manually under {extraction_path}/APAC, /EMEA, /AMER")


✓ Archive '/content/Archive.zip' extracted to '/content/data'


---
## Step 2 — Imports

In [3]:
from __future__ import annotations

import gc
import hashlib
import io
import json
import pickle
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import faiss
import numpy as np
import torch

# Explicitly import torchvision before Transformers.
# Reason: if torch and torchvision are mismatched, this gives a clear error here
# instead of a confusing "Could not import PreTrainedModel" later.
try:
    import torchvision
except Exception as e:
    raise RuntimeError(
        "torchvision failed to import. This usually means torch and torchvision "
        "versions do not match. Re-run the install cell, restart the Colab runtime, "
        "then run this import cell again."
    ) from e

from PIL import Image
from tqdm.auto import tqdm

import fitz
import pymupdf4llm

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import CrossEncoder
from transformers import AutoModel, AutoProcessor, BitsAndBytesConfig

# Official Qwen2.5-VL class name.
try:
    from transformers import Qwen2_5_VLForConditionalGeneration
except ImportError as e:
    raise ImportError(
        "Qwen2.5-VL requires a recent Transformers version. "
        "Run the install cell again, restart the runtime, then retry."
    ) from e

print("✓ All imports done")
print(f"✓ torch: {torch.__version__}")
print(f"✓ torchvision: {torchvision.__version__}")
print(f"✓ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


/tmp/ipykernel_3536/2601670144.py:36: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✓ All imports done
✓ torch: 2.7.0+cu126
✓ torchvision: 0.22.0+cu126
✓ GPU available: True
  GPU: Tesla T4
  Memory: 15.6 GB


---
## Step 3 — Settings

In [4]:
@dataclass
class Settings:
    # Persistent Google Drive paths.
    # Reason: /content is temporary in Colab; Drive survives runtime restarts.
    data_dir:    Path = Path("/content/gdrive/MyDrive/RAG_data/data")
    storage_dir: Path = Path("/content/gdrive/MyDrive/RAG_data/storage")
    images_dir:  Path = Path("/content/gdrive/MyDrive/RAG_data/storage/images")
    page_images_dir: Path = Path("/content/gdrive/MyDrive/RAG_data/storage/page_images")

    # Persistent cache for semantic VLM image captions.
    # Reason: captioning images with a VLM is expensive; cache once, reuse many times.
    image_caption_cache_path: Path = Path("/content/gdrive/MyDrive/RAG_data/storage/image_captions.json")

    # Persistent FAISS index for true visual image embeddings.
    # Reason: this stores vectors created from actual image pixels, not just captions.
    visual_image_index_dir: Path = Path("/content/gdrive/MyDrive/RAG_data/storage/visual_image_faiss")

    # Regions
    geos: Tuple[str, ...] = ("APAC", "EMEA", "AMER")

    # Chunking
    chunk_size:    int = 800
    chunk_overlap: int = 100

    # Retrieval
    # Slightly larger k helps multimodal RAG because image captions compete with text chunks.
    top_k:             int             = 2   # FAST MODE: fewer chunks = faster final answer
    retrieval_fetch_k: int             = 6   # FAST MODE: fewer FAISS candidates = less reranking/context work
    score_threshold:   Optional[float] = 1.2
    bm25_top_k:        int             = 3

    # Speed controls
    # Reason: most latency comes from cross-encoder reranking, visual-query encoding,
    # and final Qwen generation. Fast mode skips expensive steps unless they are useful.
    fast_mode: bool = True
    use_reranker: bool = False                 # False = much faster; True = better ranking quality
    use_visual_retrieval_only_for_visual_questions: bool = True


    # Embedding
    embedding_model:  str = "BAAI/bge-large-en-v1.5"
    bge_query_prefix: str = "Represent this sentence for searching relevant passages: "

    # Reranker
    reranker_model: str = "BAAI/bge-reranker-v2-m3"

    # VLM — 7B is a practical Colab choice with 4-bit quantization.
    vlm_model: str = "Qwen/Qwen2.5-VL-7B-Instruct"

    # Qwen2.5-VL image token budget.
    # Reason: higher pixels improve chart/table reading but cost more VRAM/time.
    vlm_min_pixels: int = 256 * 28 * 28
    vlm_max_pixels: int = 512 * 28 * 28   # lower image tokens; safer on 15GB Colab GPUs

    # Generation
    max_new_tokens:    int   = 128         # FAST MODE: generation speed is mostly proportional to output tokens
    image_max_tokens:  int   = 700
    do_sample:         bool  = False
    temperature:       float = 0.7

    # Image extraction/indexing
    min_image_px: int = 100
    enable_image_caption_indexing: bool = True
    refresh_image_caption_cache: bool = False
    max_images_to_caption: Optional[int] = None  # set e.g. 20 for a quick test run
    max_images_in_answer: int = 1                # keep 0 for fastest; 1 only for visual questions
    answer_image_max_side: int = 896             # resize final-answer images before sending to Qwen
    auto_use_images_only_for_visual_questions: bool = True
    direct_vision_answer_enabled: bool = True


    # Full-page screenshot indexing.
    # Reason: many PDF charts/tables are vector drawings, not embedded raster images.
    # Rendering the page catches those visuals and preserves layout around the answer.
    enable_page_screenshot_indexing: bool = True
    page_render_dpi: int = 100                   # lower DPI reduces page screenshot size and VRAM
    page_image_max_side: int = 1200              # prevents huge page screenshots from wasting memory
    max_pages_to_render_per_pdf: Optional[int] = None  # set e.g. 5 for a quick test run

    # True visual embedding retrieval.
    # Reason: caption search is useful, but image vectors let retrieval use layout/shape/pixels.
    enable_visual_embedding_indexing: bool = True
    image_vector_model: str = "google/siglip2-base-patch16-224"
    image_embedding_device: str = "cpu"             # use "cuda" only if your GPU has enough free VRAM
    image_embedding_batch_size: int = 8
    visual_top_k: int = 2
    visual_score_threshold: Optional[float] = None  # keep None first; tune later after seeing scores
    max_visual_images_to_index: Optional[int] = None # set e.g. 50 for a quick test run

    # Text filters
    min_page_chars:    int = 40
    max_chars_in_prompt: int = 800               # FAST MODE: shorter prompt = faster prefill + lower VRAM

    # Evaluation
    eval_n_per_geo:  int  = 10
    eval_seed:       int  = 42


settings = Settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)
settings.storage_dir.mkdir(parents=True, exist_ok=True)
settings.images_dir.mkdir(parents=True, exist_ok=True)
settings.page_images_dir.mkdir(parents=True, exist_ok=True)
settings.visual_image_index_dir.mkdir(parents=True, exist_ok=True)

print("✓ Settings ready")
print(f"   VLM         : {settings.vlm_model}")
print(f"   Data dir    : {settings.data_dir}")
print(f"   Storage dir : {settings.storage_dir}")
print(f"   Image cache : {settings.image_caption_cache_path}")
print(f"   Visual index: {settings.visual_image_index_dir}")


✓ Settings ready
   VLM         : Qwen/Qwen2.5-VL-7B-Instruct
   Data dir    : /content/gdrive/MyDrive/RAG_data/data
   Storage dir : /content/gdrive/MyDrive/RAG_data/storage
   Image cache : /content/gdrive/MyDrive/RAG_data/storage/image_captions.json
   Visual index: /content/gdrive/MyDrive/RAG_data/storage/visual_image_faiss


### Fast mode vs quality mode

This notebook now defaults to **fast mode**.

Why: final Qwen generation, cross-encoder reranking, and visual image-vector retrieval are the slowest steps. Fast mode keeps text/caption retrieval, but skips expensive visual retrieval and reranking unless the question is visual.

Use fast mode for summaries and normal Q&A. Use quality mode when you need best source ranking or visual/chart questions.

In [5]:
# Optional runtime switches

def set_fast_mode():
    """Fastest useful settings for normal text summaries and Q&A."""
    settings.fast_mode = True
    settings.use_reranker = False
    settings.use_visual_retrieval_only_for_visual_questions = True
    settings.direct_vision_answer_enabled = True  # still used for visual questions only
    settings.top_k = 2
    settings.retrieval_fetch_k = 6
    settings.bm25_top_k = 3
    settings.visual_top_k = 2
    settings.max_new_tokens = 128
    settings.max_chars_in_prompt = 800
    settings.max_images_in_answer = 1
    print("✓ Fast mode enabled")

def set_quality_mode():
    """Higher quality settings; slower because reranking and more context are enabled."""
    settings.fast_mode = False
    settings.use_reranker = True
    settings.use_visual_retrieval_only_for_visual_questions = False
    settings.top_k = 4
    settings.retrieval_fetch_k = 16
    settings.bm25_top_k = 5
    settings.visual_top_k = 4
    settings.max_new_tokens = 256
    settings.max_chars_in_prompt = 1400
    settings.max_images_in_answer = 1
    print("✓ Quality mode enabled. Re-run the reranker loading cell if reranker was skipped earlier.")

set_fast_mode()

✓ Fast mode enabled


---
## Step 4 — Helper functions

In [6]:
def normalize_geo(value: str) -> str:
    value = value.strip().upper()
    if value not in settings.geos:
        raise ValueError(f"Unknown region: {value}")
    return value

def detect_geos_from_query(query: str) -> List[str]:
    text = query.upper()
    return [g for g in settings.geos if re.search(rf"\\b{g}\\b", text)]

def make_doc_id(path: Path) -> str:
    return hashlib.md5(str(path.resolve()).encode()).hexdigest()

def clean_text(text: str) -> str:
    text = text.replace("\\x00", " ")
    text = re.sub(r"[ \\t]+", " ", text)
    text = re.sub(r"\\n{3,}", "\\n\\n", text)
    return text.strip()

def page_has_usable_text(text: str) -> bool:
    return len(clean_text(text)) >= settings.min_page_chars

def build_chunk_id(source: str, page: Optional[int], idx: int) -> str:
    return hashlib.md5(f"{source}|{page}|{idx}".encode()).hexdigest()

def make_image_save_path(pdf_path: Path, page_index: int, img_index: int) -> Path:
    uid = hashlib.md5(
        f"{pdf_path.resolve()}|{page_index}|{img_index}".encode()
    ).hexdigest()
    return settings.images_dir / f"{uid}.png"

def make_page_image_save_path(pdf_path: Path, page_index: int) -> Path:
    """Create a stable path for a rendered full-page screenshot.

    Why separate folder:
    - Embedded images and full-page screenshots are different evidence types.
    - Keeping them separate makes debugging and cleanup easier.
    """
    uid = hashlib.md5(
        f"{pdf_path.resolve()}|page-screenshot|{page_index}".encode()
    ).hexdigest()
    return settings.page_images_dir / f"{uid}.png"


def preprocess_pdf_image(pil_img: Image.Image, max_side: Optional[int] = None) -> Image.Image:
    """Light preprocessing for images extracted/rendered from PDFs.

    Why this is intentionally simple:
    - Qwen2.5-VL and SigLIP2 processors already resize/normalize for the model.
    - We only standardize color and prevent extremely large files from wasting memory.
    - Avoid heavy sharpening/thresholding because it can distort charts and visual embeddings.
    """
    # Convert transparent/CMYK/grayscale/palette images into normal RGB.
    if pil_img.mode in ("RGBA", "LA"):
        background = Image.new("RGB", pil_img.size, "white")
        alpha = pil_img.getchannel("A")
        background.paste(pil_img.convert("RGB"), mask=alpha)
        img = background
    else:
        img = pil_img.convert("RGB")

    # Downscale only when the image is very large.
    # Reason: protects Colab RAM/VRAM while preserving readable charts and text.
    if max_side is not None:
        w, h = img.size
        largest_side = max(w, h)
        if largest_side > max_side:
            scale = max_side / largest_side
            new_size = (max(1, int(w * scale)), max(1, int(h * scale)))
            img = img.resize(new_size, Image.LANCZOS)

    return img


def render_and_save_page_image(pdf_path: Path, page_index: int) -> Optional[Dict]:
    """Render a full PDF page as an image and save it.

    Why:
    - page.get_images() only extracts embedded raster images.
    - Many PDF charts/tables/diagrams are vector drawings, so they are invisible to image extraction.
    - A rendered page screenshot gives the visual retriever and VLM the full page layout.
    """
    if not settings.enable_page_screenshot_indexing:
        return None

    try:
        img_save_path = make_page_image_save_path(pdf_path, page_index)

        # Reuse an existing page screenshot when available.
        # Reason: rendering pages repeatedly is slow and wastes Drive writes.
        if img_save_path.exists():
            pil_img = Image.open(img_save_path).convert("RGB")
        else:
            zoom = settings.page_render_dpi / 72.0
            matrix = fitz.Matrix(zoom, zoom)

            with fitz.open(str(pdf_path)) as doc:
                if page_index < 0 or page_index >= len(doc):
                    return None
                page = doc[page_index]
                pix = page.get_pixmap(matrix=matrix, alpha=False)

            pil_img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            pil_img = preprocess_pdf_image(pil_img, max_side=settings.page_image_max_side)
            pil_img.save(str(img_save_path), format="PNG", optimize=True)

        w, h = pil_img.size
        caption = (
            f"Uncaptioned full-page screenshot from {pdf_path.name}, page {page_index}. "
            f"Size: {w}x{h}px. This page image captures vector charts, tables, diagrams, "
            "and layout that embedded-image extraction may miss."
        )

        return {
            "caption": caption,
            "image_path": str(img_save_path),
            "width": w,
            "height": h,
            "image_kind": "page_screenshot",
        }

    except Exception as e:
        print(f"  ⚠  Page screenshot render error for {pdf_path.name} page {page_index}: {e}")
        return None


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("✓ Helper functions ready")

✓ Helper functions ready


---
## Step 5 — Text and table detection

In [7]:
def is_table_line(line: str) -> bool:
    s = line.strip()
    return s.startswith("|") and s.count("|") >= 2

# This function splits one page’s markdown into text blocks and table blocks.
def split_page_into_segments(markdown_text: str) -> List[Dict]:
    segments: List[Dict] = []
    buffer: List[str] = []
    in_table = False

    for line in markdown_text.split("\\n"):
        if is_table_line(line):
            if not in_table:
                content = "\\n".join(buffer).strip()
                if content:
                    segments.append({"type": "text", "content": content})
                buffer = []
                in_table = True
            buffer.append(line)
        else:
            if in_table:
                content = "\\n".join(buffer).strip()
                if content:
                    segments.append({"type": "table", "content": content})
                buffer = []
                in_table = False
            buffer.append(line)

    if buffer:
        content = "\\n".join(buffer).strip()
        if content:
            segments.append({"type": "table" if in_table else "text", "content": content})
    return segments

def extract_json_blocks(text: str) -> Tuple[List[str], str]:
    pattern = re.compile(r"\\{[^{}]*\\}", re.DOTALL)
    found: List[str] = []

    def replacer(m):
        raw = m.group(0)
        try:
            json.loads(raw)
            found.append(raw)
            return " "
        except (json.JSONDecodeError, ValueError):
            return raw

    cleaned = pattern.sub(replacer, text).strip()
    return found, cleaned

def json_to_sentences(json_str: str) -> str:
    try:
        data = json.loads(json_str)
        if isinstance(data, dict):
            parts = [f"{str(k).replace('_', ' ')}: {v}" for k, v in data.items()]
            return ". ".join(parts) + "."
        if isinstance(data, list):
            return " | ".join(
                json_to_sentences(json.dumps(i)) if isinstance(i, dict) else str(i)
                for i in data
            )
        return str(data)
    except (json.JSONDecodeError, ValueError):
        return json_str

print("✓ Text detection ready")

✓ Text detection ready


---
## Step 6 — Image extraction

In [8]:
def extract_and_save_images(pdf_path: Path, page_index: int) -> List[Dict]:
    """Extract embedded images from a PDF page and save them as PNG files.

    Important: the first caption is only a cheap placeholder. Later, after Qwen2.5-VL
    is loaded, we replace this with a semantic VLM caption before indexing.
    """
    results: List[Dict] = []
    try:
        doc = fitz.open(str(pdf_path))
        page = doc[page_index]

        for img_idx, img_info in enumerate(page.get_images(full=True)):
            xref = img_info[0]
            try:
                base_img = doc.extract_image(xref)
                w = base_img.get("width", 0)
                h = base_img.get("height", 0)

                # Skip tiny logos/icons; they add noise to the image index.
                if w < settings.min_image_px or h < settings.min_image_px:
                    continue

                pil_img = Image.open(io.BytesIO(base_img["image"]))
                pil_img = preprocess_pdf_image(pil_img, max_side=settings.page_image_max_side)
                w, h = pil_img.size

                img_save_path = make_image_save_path(pdf_path, page_index, img_idx)
                pil_img.save(str(img_save_path), format="PNG", optimize=True)

                caption = (
                    f"Uncaptioned image from {pdf_path.name}, page {page_index}. "
                    f"Size: {w}x{h}px. This placeholder will be replaced by a VLM caption before indexing."
                )

                results.append({
                    "caption": caption,
                    "image_path": str(img_save_path),
                    "width": w,
                    "height": h,
                    "image_kind": "embedded_image",
                })

            except Exception:
                continue

        doc.close()
    except Exception as e:
        print(f"  ⚠  Image extraction error: {e}")

    return results

print("✓ Image extraction ready")


✓ Image extraction ready


---
## Step 7 — PDF loader

In [9]:
def build_documents_from_pdf(pdf_path: Path, geo: str) -> List[Document]:
    doc_id = make_doc_id(pdf_path)
    all_docs: List[Document] = []

    try:
        pages = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
    except Exception as e:
        print(f"  ✗ Cannot parse {pdf_path.name}: {e}")
        return []

    for page_idx_in_doc, page_data in enumerate(pages):
        page_num = page_data.get("metadata", {}).get("page", page_idx_in_doc)
        page_index = page_idx_in_doc  # PyMuPDF uses zero-based page indexes for rendering/extraction.
        page_text = page_data.get("text", "") or ""

        base_meta = {
            "geo": geo,
            "source": str(pdf_path),
            "doc_id": doc_id,
            "file_name": pdf_path.name,
            "page": page_num,
        }

        # 1A) Render the full page as an optional visual document.
        # Reason: charts/tables may be vector drawings, so they will not appear in page.get_images().
        if (
            settings.enable_page_screenshot_indexing
            and (
                settings.max_pages_to_render_per_pdf is None
                or page_idx_in_doc < settings.max_pages_to_render_per_pdf
            )
        ):
            page_item = render_and_save_page_image(pdf_path, page_index)
            
            if page_item is not None:
                all_docs.append(Document(
                    page_content=page_item["caption"],
                    metadata={
                        **base_meta,
                        "element_type": "vlm_page_image",
                        "image_path": page_item["image_path"],
                        "image_width": page_item.get("width"),
                        "image_height": page_item.get("height"),
                        "image_kind": "page_screenshot",
                        "image_caption_enriched": False,
                    },
                ))

        # 1B) Extract embedded raster images from every page.
        # Reason: charts/diagrams/screenshots often contain information that text parsers miss.
        for item in extract_and_save_images(pdf_path, page_index):
            all_docs.append(Document(
                page_content=item["caption"],
                metadata={
                    **base_meta,
                    "element_type": "vlm_image",
                    "image_path": item["image_path"],
                    "image_width": item.get("width"),
                    "image_height": item.get("height"),
                    "image_kind": "embedded_image",
                    "image_caption_enriched": False,
                },
            ))

        # 2) Process text/tables/json.
        if not page_has_usable_text(page_text):
            continue

        for seg in split_page_into_segments(page_text):
            content = seg["content"]
            if not content.strip():
                continue

            if seg["type"] == "table":
                all_docs.append(Document(
                    page_content=clean_text(content),
                    metadata={**base_meta, "element_type": "table"},
                ))
            else:
                json_blocks, remaining = extract_json_blocks(content)

                for raw_json in json_blocks:
                    readable = json_to_sentences(raw_json)
                    if len(readable) >= settings.min_page_chars:
                        all_docs.append(Document(
                            page_content=readable,
                            metadata={**base_meta, "element_type": "json"},
                        ))

                remaining = clean_text(remaining)
                if page_has_usable_text(remaining):
                    all_docs.append(Document(
                        page_content=remaining,
                        metadata={**base_meta, "element_type": "text"},
                    ))

    return all_docs

print("✓ PDF loader ready")


✓ PDF loader ready


---
## Step 8 — Load all PDFs

In [10]:
geo_page_docs: Dict[str, List[Document]] = {}

for geo in settings.geos:
    print(f"\n── {geo} ──")
    geo_dir = settings.data_dir / geo

    if not geo_dir.exists():
        print(f"  ⚠  Not found: {geo_dir}")
        geo_page_docs[geo] = []
        continue

    pdf_paths = sorted(geo_dir.rglob("*.pdf"))
    if not pdf_paths:
        print(f"  ⚠  No PDFs found")
        geo_page_docs[geo] = []
        continue

    all_docs = []
    visual_count = 0

    for i, pdf_path in enumerate(pdf_paths, 1):
        size_mb = pdf_path.stat().st_size / (1024 * 1024)
        if size_mb > 100:
            print(f"  [{i}/{len(pdf_paths)}] {pdf_path.name} — SKIP (too large)")
            continue

        print(f"  [{i}/{len(pdf_paths)}] {pdf_path.name}...", end=" ", flush=True)

        try:
            docs = build_documents_from_pdf(pdf_path, geo)
            all_docs.extend(docs)

            counts = {}
            for d in docs:
                t = d.metadata.get("element_type", "?")
                counts[t] = counts.get(t, 0) + 1
                if t in ("vlm_image", "vlm_page_image"):
                    visual_count += 1

            print(f"OK  {counts}")

        except Exception as e:
            print(f"SKIP ({type(e).__name__})")
            continue

        clear_memory()

    geo_page_docs[geo] = all_docs
    print(f"  Done: {len(all_docs)} elements, {visual_count} visual files")

saved_images = list(settings.images_dir.glob("*.png"))
saved_page_images = list(settings.page_images_dir.glob("*.png"))
print(f"\n✓ PDFs loaded — {len(saved_images)} embedded images and {len(saved_page_images)} page screenshots saved to disk")


── APAC ──
  [1/5] copy-protected.pdf... === Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.

OK  {'vlm_page_image': 2, 'text': 2}
  [2/5] embedded-images-tables.pdf... === Document parser messages ===
                                                            Using Tesseract for OCR processing.
OCR on page.number=0/1.

OK  {'vlm_page_image': 1, 'vlm_image': 1, 'text': 1}
  [3/5] embedded-images.pdf... === Document parser messages ===
                                                                                                                        Using Tesseract for OCR processing.
OCR on page.number=0/1.

OK  {'vlm_page_image': 1, 'vlm_image': 3, 'text': 1}
  [4/5] embedded-link.pdf... === Document parser messages ===
                                                                                                                                                                                    Using Tesseract for OCR processing.

OK  {

---
## Step 9 — Chunk documents

In [11]:
def chunk_documents(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=settings.chunk_size,
        chunk_overlap=settings.chunk_overlap,
    )
    chunked: List[Document] = []
    idx = 0

    for doc in docs:
        elem_type = doc.metadata.get("element_type", "text")
        sub_chunks = splitter.split_documents([doc]) if elem_type == "text" else [doc]

        for chunk in sub_chunks:
            chunk.metadata["chunk_id"] = build_chunk_id(
                chunk.metadata.get("source", ""),
                chunk.metadata.get("page"),
                idx,
            )
            chunked.append(chunk)
            idx += 1

    return chunked

geo_chunk_docs: Dict[str, List[Document]] = {}
for geo in settings.geos:
    chunks = chunk_documents(geo_page_docs[geo])
    geo_chunk_docs[geo] = chunks
    breakdown = {}
    for c in chunks:
        t = c.metadata.get("element_type", "?")
        breakdown[t] = breakdown.get(t, 0) + 1
    print(f"{geo}: {len(chunks)} chunks  {breakdown}")

print("\n✓ Chunking complete")

APAC: 70 chunks  {'vlm_page_image': 7, 'text': 35, 'vlm_image': 28}
EMEA: 116 chunks  {'vlm_page_image': 17, 'text': 95, 'table': 3, 'vlm_image': 1}
AMER: 14 chunks  {'vlm_page_image': 3, 'vlm_image': 2, 'text': 9}

✓ Chunking complete


---
## Step 10 — Build indexes

In [12]:
class BGEEmbeddings(HuggingFaceEmbeddings):
    def embed_query(self, text: str) -> List[float]:
        # BGE retrieval models work better when user queries use the recommended query instruction.
        return super().embed_query(settings.bge_query_prefix + text)

print(f"Loading embedding model: {settings.embedding_model}")
embeddings = BGEEmbeddings(
    model_name=settings.embedding_model,
    encode_kwargs={"normalize_embeddings": True},
)

reranker = None
if settings.use_reranker:
    print("Loading reranker...")
    reranker = CrossEncoder(settings.reranker_model, max_length=512)
else:
    print("Skipping reranker because settings.use_reranker=False. This is faster; set True for quality mode.")

def geo_index_path(geo: str) -> Path:
    return settings.storage_dir / f"faiss_{geo.lower()}"

def build_and_save_indexes(geo_chunks, embed_model):
    """Build one FAISS + BM25 pair per region.

    Reason:
    - FAISS finds semantic matches.
    - BM25 catches exact words, IDs, product names, and region-specific terms.
    - The reranker later chooses the best final chunks from both.
    """
    faiss_stores, bm25_stores = {}, {}

    for geo, chunks in geo_chunks.items():
        if not chunks:
            print(f"⚠  Skipping {geo}: no chunks")
            continue

        save_path = geo_index_path(geo)
        save_path.mkdir(parents=True, exist_ok=True)

        print(f"\n── {geo} ({len(chunks)} chunks) ──")

        print("  Building FAISS...")
        store = FAISS.from_documents(chunks, embed_model)
        store.save_local(str(save_path))
        faiss_stores[geo] = store

        print("  Building BM25...")
        bm25 = BM25Retriever.from_documents(chunks)
        bm25.k = settings.bm25_top_k
        bm25_stores[geo] = bm25

        with open(save_path / "chunks.pkl", "wb") as f:
            pickle.dump(chunks, f)

        print("  ✓ FAISS + BM25 ready")
        clear_memory()

    return faiss_stores, bm25_stores

print("✓ Embeddings and index builder ready")
print("ℹ Indexes will be built after VLM image captions are added.")


Loading embedding model: BAAI/bge-large-en-v1.5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Skipping reranker because settings.use_reranker=False. This is faster; set True for quality mode.
✓ Embeddings and index builder ready
ℹ Indexes will be built after VLM image captions are added.


---
## Step 11 — Load VLM (7B for Colab)

In [13]:
import os
from pathlib import Path

# Use Drive cache so the model is not downloaded every Colab session.
cache_dir = Path("/content/gdrive/MyDrive/model_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(cache_dir)

print(f"Loading VLM from cache: {cache_dir}")
print("First run downloads the model. Later runs reuse the Drive cache.\n")

clear_memory()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    settings.vlm_model,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    cache_dir=str(cache_dir),
)

# min_pixels/max_pixels control visual token count.
# Reason: this keeps image understanding strong while avoiding Colab VRAM spikes.
vlm_processor = AutoProcessor.from_pretrained(
    settings.vlm_model,
    min_pixels=settings.vlm_min_pixels,
    max_pixels=settings.vlm_max_pixels,
    cache_dir=str(cache_dir),
)

print("✓ VLM loaded")
print(f"  Device: {next(vlm.parameters()).device}")
print("  Mode  : 4-bit quantized")


Loading VLM from cache: /content/gdrive/MyDrive/model_cache
First run downloads the model. Later runs reuse the Drive cache.



Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✓ VLM loaded
  Device: cuda:0
  Mode  : 4-bit quantized


---
## Step 12 — VLM image description (cached)

In [14]:
def load_image_caption_cache() -> Dict[str, str]:
    """Load persistent image-caption cache from Drive."""
    if settings.image_caption_cache_path.exists():
        try:
            with open(settings.image_caption_cache_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_image_caption_cache() -> None:
    """Save persistent image-caption cache to Drive."""
    settings.image_caption_cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(settings.image_caption_cache_path, "w", encoding="utf-8") as f:
        json.dump(_image_caption_cache, f, ensure_ascii=False, indent=2)

_image_caption_cache: Dict[str, str] = load_image_caption_cache()
print(f"✓ Loaded {len(_image_caption_cache)} cached image captions")

def describe_image_with_vlm(image_path: str, refresh: Optional[bool] = None) -> str:
    """Create a structured VLM caption for one image.

    Reason for structured captions:
    - FAISS/BM25 cannot search pixels.
    - A caption containing visible text, chart axes, numbers, and trends makes the image searchable.
    - The caption is cached because VLM inference is slower than text embedding.
    """
    refresh = settings.refresh_image_caption_cache if refresh is None else refresh

    if (not refresh) and image_path in _image_caption_cache:
        return _image_caption_cache[image_path]

    img_path_obj = Path(image_path)
    if not img_path_obj.exists():
        return "[Image file not found]"

    try:
        pil_img = Image.open(image_path).convert("RGB")

        prompt = (
            "You are creating a search-index caption for multimodal RAG.\n"
            "Analyze the image carefully and return concise structured notes.\n\n"
            "Include:\n"
            "1. content_type: chart/table/diagram/screenshot/photo/other\n"
            "2. visible_text: all important readable text and labels\n"
            "3. key_values: important numbers, percentages, dates, names, axes, legends\n"
            "4. relationships: trends, comparisons, flows, hierarchy, or layout meaning\n"
            "5. retrieval_summary: one short paragraph describing when this image is relevant\n\n"
            "Do not invent information. If text is unreadable, say unreadable."
        )

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": pil_img},
                {"type": "text", "text": prompt},
            ],
        }]

        text_input = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = vlm_processor(
            text=[text_input],
            images=[pil_img],
            return_tensors="pt",
            padding=True,
        ).to(vlm.device)

        with torch.no_grad():
            output_ids = vlm.generate(
                **inputs,
                max_new_tokens=settings.image_max_tokens,
                do_sample=False,
                temperature=settings.temperature,
            )

        generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
        description = vlm_processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0].strip()

        _image_caption_cache[image_path] = description
        save_image_caption_cache()
        return description

    except Exception as e:
        error_msg = f"[Image captioning error: {type(e).__name__}: {str(e)[:120]}]"
        _image_caption_cache[image_path] = error_msg
        save_image_caption_cache()
        return error_msg

print("✓ VLM image captioning ready")


✓ Loaded 58 cached image captions
✓ VLM image captioning ready


---
## Step 12B — Advanced semantic visual indexing

This step upgrades the notebook from **visual extraction** to **visual retrieval**.

Why we use it:

- A raw image path is not searchable.
- Generic text like “chart or diagram” is too weak for retrieval.
- A VLM caption turns each embedded image or rendered page screenshot into searchable evidence with labels, values, trends, and meaning.
- The caption is indexed with the same FAISS + BM25 flow, so no separate complex image database is required.

Why page screenshots are included:

- `page.get_images()` only extracts embedded raster images.
- Many PDF charts/tables/diagrams are vector drawings and will not be extracted as images.
- Rendering the full page preserves layout and lets Qwen/SigLIP see what the text parser missed.

This is a practical, low-rework version of modern multimodal document RAG.


In [15]:
def build_image_index_text(doc: Document, caption: str) -> str:
    """Convert image metadata + VLM caption into searchable text."""
    meta = doc.metadata
    return clean_text(f"""
[VISUAL EVIDENCE]
Evidence type: {meta.get('element_type', '?')}
Image kind: {meta.get('image_kind', 'embedded_image')}
Region: {meta.get('geo', '?')}
File: {meta.get('file_name', '?')}
Page: {meta.get('page', '?')}
Image size: {meta.get('image_width', '?')}x{meta.get('image_height', '?')}

VLM semantic caption:
{caption}
""")

def enrich_image_documents_for_indexing(geo_chunks: Dict[str, List[Document]]) -> Dict[str, List[Document]]:
    """Replace generic image placeholders with Qwen2.5-VL semantic captions.

    This is the main image-indexing upgrade.
    After this function runs, FAISS and BM25 can retrieve images by their actual content.
    """
    if not settings.enable_image_caption_indexing:
        print("ℹ Image caption indexing disabled; using placeholder image captions.")
        return geo_chunks

    total_seen = 0
    total_enriched = 0

    for geo, docs in geo_chunks.items():
        print(f"\n── Enriching image captions for {geo} ──")
        image_docs = [
            d for d in docs
            if d.metadata.get("element_type") in ("vlm_image", "vlm_page_image")
        ]
        print(f"  Found {len(image_docs)} visual documents: embedded images + page screenshots")

        for doc in tqdm(image_docs, desc=f"Captioning {geo}"):
            if settings.max_images_to_caption is not None and total_seen >= settings.max_images_to_caption:
                print("  ℹ Reached max_images_to_caption limit")
                break

            img_path = doc.metadata.get("image_path")
            if not img_path or not Path(img_path).exists():
                continue

            caption = describe_image_with_vlm(img_path)
            doc.page_content = build_image_index_text(doc, caption)
            doc.metadata["image_caption_enriched"] = True
            total_seen += 1
            total_enriched += 1

        print(f"  ✓ Enriched {sum(d.metadata.get('image_caption_enriched', False) for d in image_docs)} visual documents in {geo}")

    save_image_caption_cache()
    print(f"\n✓ Semantic visual captions ready: {total_enriched} embedded images/page screenshots processed")
    return geo_chunks

# 1) Add VLM semantic captions to image docs.
geo_chunk_docs = enrich_image_documents_for_indexing(geo_chunk_docs)

# 2) Build indexes AFTER image docs have real semantic captions.
geo_vectorstores, geo_bm25_stores = build_and_save_indexes(geo_chunk_docs, embeddings)
print("\n✓ Multimodal indexes built and saved to Google Drive")



── Enriching image captions for APAC ──
  Found 35 visual documents: embedded images + page screenshots


Captioning APAC:   0%|          | 0/35 [00:00<?, ?it/s]

  ✓ Enriched 35 visual documents in APAC

── Enriching image captions for EMEA ──
  Found 18 visual documents: embedded images + page screenshots


Captioning EMEA:   0%|          | 0/18 [00:00<?, ?it/s]

  ✓ Enriched 18 visual documents in EMEA

── Enriching image captions for AMER ──
  Found 5 visual documents: embedded images + page screenshots


Captioning AMER:   0%|          | 0/5 [00:00<?, ?it/s]

  ✓ Enriched 5 visual documents in AMER

✓ Semantic visual captions ready: 58 embedded images/page screenshots processed

── APAC (70 chunks) ──
  Building FAISS...
  Building BM25...
  ✓ FAISS + BM25 ready

── EMEA (116 chunks) ──
  Building FAISS...
  Building BM25...
  ✓ FAISS + BM25 ready

── AMER (14 chunks) ──
  Building FAISS...
  Building BM25...
  ✓ FAISS + BM25 ready

✓ Multimodal indexes built and saved to Google Drive


---
## Step 12C — True visual embedding index

Caption-based visual indexing is useful, but it still searches **text about the visual**.

This step adds a separate visual-vector index:

```text
embedded image pixels      → SigLIP2 image embedding → FAISS visual index
rendered page screenshot   → SigLIP2 image embedding → FAISS visual index
user query                 → SigLIP2 text embedding  → search visual vectors
```

Why we use it:

- It makes retrieval more genuinely multimodal.
- It can find visually relevant charts, screenshots, diagrams, and page layouts even when the caption is incomplete.
- It is low rework because it adds a second visual-only FAISS index beside your existing text/caption FAISS index.


In [16]:
def _clone_doc_with_metadata(doc: Document, extra_meta: Dict) -> Document:
    """Return a safe copy of a Document with extra metadata.

    Reason: the same image document can be found by FAISS, BM25, or visual search.
    Copying avoids accidentally overwriting metadata in the shared source document.
    """
    return Document(
        page_content=doc.page_content,
        metadata={**doc.metadata, **extra_meta},
    )

def get_image_docs_for_visual_index(docs: List[Document]) -> List[Document]:
    """Keep visual documents that have real image files on disk.

    Includes both:
    - embedded PDF images
    - rendered full-page screenshots

    Reason: page screenshots catch vector charts/tables that embedded-image extraction misses.
    """
    image_docs = []
    for doc in docs:
        if doc.metadata.get("element_type") not in ("vlm_image", "vlm_page_image"):
            continue

        img_path = doc.metadata.get("image_path")
        if img_path and Path(img_path).exists():
            image_docs.append(doc)

    if settings.max_visual_images_to_index is not None:
        image_docs = image_docs[:settings.max_visual_images_to_index]

    return image_docs

visual_processor = None
visual_model = None
visual_device = None

def load_visual_embedding_model():
    """Load SigLIP2 for cross-modal image retrieval.

    Why SigLIP2:
    - It is a modern vision-language encoder.
    - It supports image-text retrieval style usage.
    - It is lighter than using a full generative VLM for every retrieval query.
    """
    global visual_processor, visual_model, visual_device

    if not settings.enable_visual_embedding_indexing:
        print("ℹ Visual embedding indexing disabled.")
        return None, None

    if visual_processor is not None and visual_model is not None:
        return visual_processor, visual_model

    requested_device = settings.image_embedding_device.lower().strip()
    if requested_device == "cuda" and not torch.cuda.is_available():
        print("⚠ CUDA requested for image embeddings, but no GPU is available. Using CPU.")
        requested_device = "cpu"

    visual_device = torch.device(requested_device)
    dtype = torch.float16 if visual_device.type == "cuda" else torch.float32

    print(f"Loading visual embedding model: {settings.image_vector_model}")
    print(f"  Device: {visual_device}")
    print("  Reason: this model embeds image pixels and text queries into the same vector space.")

    visual_processor = AutoProcessor.from_pretrained(
        settings.image_vector_model,
        cache_dir=str(cache_dir),
    )

    # Some Transformers versions use torch_dtype, newer versions may prefer dtype.
    try:
        visual_model = AutoModel.from_pretrained(
            settings.image_vector_model,
            torch_dtype=dtype,
            cache_dir=str(cache_dir),
        )
    except TypeError:
        visual_model = AutoModel.from_pretrained(
            settings.image_vector_model,
            dtype=dtype,
            cache_dir=str(cache_dir),
        )

    visual_model = visual_model.to(visual_device).eval()
    return visual_processor, visual_model

def _to_visual_device(batch: Dict) -> Dict:
    """Move processor tensors to the image embedding device."""
    return {
        k: (v.to(visual_device) if hasattr(v, "to") else v)
        for k, v in batch.items()
    }

def _siglip_output_to_tensor(output):
    """Convert SigLIP/SigLIP2 outputs into a 2D feature tensor.

    Why this helper exists:
    - Some Transformers versions return a plain tensor from get_image_features/get_text_features.
    - Other versions return BaseModelOutputWithPooling.
    - For retrieval we need the pooled 2D embedding: [batch_size, embedding_dim].
    """
    if isinstance(output, torch.Tensor):
        return output

    # SigLIP2 docs list BaseModelOutputWithPooling for get_*_features.
    # pooler_output is the correct compact vector for similarity search.
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output

    # Some multimodal outputs expose already projected embeddings.
    for attr in ("image_embeds", "text_embeds"):
        if hasattr(output, attr) and getattr(output, attr) is not None:
            return getattr(output, attr)

    # return_dict=False may return a tuple/list. Prefer the 2D tensor if present.
    if isinstance(output, (tuple, list)):
        for item in output:
            if isinstance(item, torch.Tensor) and item.ndim == 2:
                return item
        for item in output:
            if isinstance(item, torch.Tensor) and item.ndim == 3:
                return item[:, 0, :]

    # Last fallback: take CLS/first token from sequence output.
    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]

    raise TypeError(f"Unsupported SigLIP output type: {type(output)}")

def encode_images_with_siglip2(image_paths: List[str]) -> np.ndarray:
    """Encode image files into normalized SigLIP2 vectors."""
    load_visual_embedding_model()

    vectors = []
    batch_size = settings.image_embedding_batch_size

    for start in tqdm(range(0, len(image_paths), batch_size), desc="Image vectors"):
        batch_paths = image_paths[start:start + batch_size]
        images = []

        for p in batch_paths:
            try:
                images.append(Image.open(p).convert("RGB"))
            except Exception:
                images.append(None)

        valid_pairs = [(p, img) for p, img in zip(batch_paths, images) if img is not None]
        if not valid_pairs:
            continue

        valid_images = [img for _, img in valid_pairs]
        inputs = visual_processor(
            images=valid_images,
            return_tensors="pt",
        )
        inputs = _to_visual_device(inputs)

        with torch.no_grad():
            raw_feats = visual_model.get_image_features(**inputs)
            feats = _siglip_output_to_tensor(raw_feats)
            feats = torch.nn.functional.normalize(feats.float(), p=2, dim=-1)

        vectors.append(feats.cpu().numpy().astype("float32"))

    if not vectors:
        return np.empty((0, 0), dtype="float32")

    return np.vstack(vectors).astype("float32")

def encode_text_with_siglip2(text: str) -> np.ndarray:
    """Encode a user query into the same vector space as image embeddings."""
    load_visual_embedding_model()

    inputs = visual_processor(
        text=[text],
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    inputs = _to_visual_device(inputs)

    with torch.no_grad():
        raw_feats = visual_model.get_text_features(**inputs)
        feats = _siglip_output_to_tensor(raw_feats)
        feats = torch.nn.functional.normalize(feats.float(), p=2, dim=-1)

    return feats.cpu().numpy().astype("float32")

def build_and_save_visual_image_indexes(geo_chunks: Dict[str, List[Document]]) -> Dict:
    """Build one FAISS image-vector index per region.

    Important:
    - The existing LangChain FAISS index stores text/caption embeddings.
    - This visual FAISS index stores image-pixel embeddings.
    - The metadata still stores image_path, so we can load the real image for Qwen2.5-VL later.
    """
    if not settings.enable_visual_embedding_indexing:
        print("ℹ Visual embedding indexing disabled.")
        return {}

    load_visual_embedding_model()

    visual_indexes = {}

    for geo, docs in geo_chunks.items():
        print(f"\n── Building visual image index for {geo} ──")
        image_docs = get_image_docs_for_visual_index(docs)

        if not image_docs:
            print("  ⚠ No image docs found")
            continue

        image_paths = [d.metadata["image_path"] for d in image_docs]
        image_vectors = encode_images_with_siglip2(image_paths)

        if image_vectors.size == 0:
            print("  ⚠ No image vectors created")
            continue

        dim = image_vectors.shape[1]
        index = faiss.IndexFlatIP(dim)  # inner product = cosine similarity after normalization
        index.add(image_vectors)

        index_path = settings.visual_image_index_dir / f"{geo.lower()}_images.faiss"
        docs_path = settings.visual_image_index_dir / f"{geo.lower()}_image_docs.pkl"

        faiss.write_index(index, str(index_path))
        with open(docs_path, "wb") as f:
            pickle.dump(image_docs, f)

        visual_indexes[geo] = {
            "index": index,
            "docs": image_docs,
            "index_path": str(index_path),
            "docs_path": str(docs_path),
        }

        print(f"  ✓ {len(image_docs)} visual documents indexed")
        print(f"  Saved: {index_path.name}")

    print("\n✓ True visual image indexes ready")
    return visual_indexes

def search_visual_image_index(question: str, visual_indexes: Dict, target_geos: List[str]) -> List[Document]:
    """Retrieve image docs using actual image embeddings.

    Reason:
    - Text/caption FAISS retrieves image captions.
    - This function retrieves image pixels using a text query projected into the same vector space.
    """
    if not settings.enable_visual_embedding_indexing or not visual_indexes:
        return []

    query_vec = encode_text_with_siglip2(question)
    visual_candidates: List[Document] = []
    seen = set()

    for geo in target_geos:
        pack = visual_indexes.get(geo)
        if not pack:
            continue

        index = pack["index"]
        docs = pack["docs"]
        k = min(settings.visual_top_k, index.ntotal)

        if k <= 0:
            continue

        scores, ids = index.search(query_vec, k)

        for score, idx in zip(scores[0], ids[0]):
            if idx < 0:
                continue

            score = float(score)
            if settings.visual_score_threshold is not None and score < settings.visual_score_threshold:
                continue

            doc = docs[int(idx)]
            cid = doc.metadata.get("chunk_id", doc.metadata.get("image_path", ""))
            if cid in seen:
                continue

            seen.add(cid)
            visual_candidates.append(
                _clone_doc_with_metadata(
                    doc,
                    {
                        "retrieval_channel": "visual_embedding",
                        "visual_score": score,
                    },
                )
            )

    return visual_candidates

# Build a second, visual-document FAISS index.
# This is separate from LangChain FAISS because text embeddings and image embeddings have different dimensions/models.
visual_image_indexes = build_and_save_visual_image_indexes(geo_chunk_docs)

print("✓ Visual retrieval functions ready")


Loading visual embedding model: google/siglip2-base-patch16-224
  Device: cpu
  Reason: this model embeds image pixels and text queries into the same vector space.


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]


── Building visual image index for APAC ──


...

  ✓ 35 visual documents indexed
  Saved: apac_images.faiss

── Building visual image index for EMEA ──


Image vectors:   0%|          | 0/3 [00:00<?, ?it/s]

  ✓ 18 visual documents indexed
  Saved: emea_images.faiss

── Building visual image index for AMER ──


Image vectors:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ 5 visual documents indexed
  Saved: amer_images.faiss

✓ True visual image indexes ready
✓ Visual retrieval functions ready


---
## Step 13 — Retrieval and generation

In [17]:
SYSTEM_PROMPT = (
    "You are a sales assistant. Answer using ONLY the provided context and retrieved images. "
    "Do not use external knowledge. "
    "If the answer is not in the context or images, say: "
    "'I do not have the answer in the provided documents.'"
)


VISUAL_QUESTION_TERMS = (
    "image", "picture", "photo", "screenshot", "chart", "graph", "plot",
    "figure", "diagram", "visual", "shown", "see", "look", "table", "page"
)

def should_use_direct_images(question: str) -> bool:
    """Decide whether final answer should send actual images/pages to Qwen.

    Why:
    - Direct VLM image input is useful for charts, diagrams, screenshots, and visual questions.
    - For broad text questions like "summarize all EMEA documents", images add VRAM cost but little value.
    """
    if not settings.direct_vision_answer_enabled:
        return False

    if not settings.auto_use_images_only_for_visual_questions:
        return True

    q = question.lower()
    return any(term in q for term in VISUAL_QUESTION_TERMS)


def should_use_visual_retrieval(question: str) -> bool:
    """Decide whether to run true visual image-vector retrieval.

    Why:
    - SigLIP text encoding + FAISS image search is useful for visual/chart/page questions.
    - For plain summaries, text/caption FAISS + BM25 is usually enough and faster.
    """
    if not settings.enable_visual_embedding_indexing:
        return False

    if not settings.use_visual_retrieval_only_for_visual_questions:
        return True

    return should_use_direct_images(question)


def resize_for_vlm_answer(img: Image.Image) -> Image.Image:
    """Resize images before final Qwen answer generation to avoid CUDA OOM.

    Note:
    The Qwen processor still performs its own token-budget resizing using min_pixels/max_pixels.
    This extra resize protects Colab memory when the source is a full-page screenshot.
    """
    img = img.convert("RGB")
    w, h = img.size
    max_side = max(w, h)

    if max_side <= settings.answer_image_max_side:
        return img

    scale = settings.answer_image_max_side / max_side
    new_size = (max(1, int(w * scale)), max(1, int(h * scale)))
    return img.resize(new_size, Image.LANCZOS)


def cleanup_cuda_memory():
    """Release cached GPU memory between VLM calls."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def format_text_context(docs: List[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        content = doc.page_content[:settings.max_chars_in_prompt]
        blocks.append(
            f"[{i}] {meta.get('element_type','?')} | "
            f"{meta.get('file_name','?')} p.{meta.get('page','?')} | "
            f"channel={meta.get('retrieval_channel','reranked')}\n{content}"
        )
    return "\n\n---\n\n".join(blocks)

def load_answer_images(docs: List[Document]) -> Tuple[List[Image.Image], List[str]]:
    """Load a small number of retrieved visual files for direct VLM answering.

    These can be embedded PDF images or rendered full-page screenshots.

    Reason: retrieval finds the likely useful visual evidence, but direct image input lets Qwen inspect
    the actual chart/diagram/page layout again before writing the final answer.
    """
    pil_images: List[Image.Image] = []
    labels: List[str] = []
    seen_paths = set()

    for doc in docs:
        if len(pil_images) >= settings.max_images_in_answer:
            break

        img_path = doc.metadata.get("image_path")
        if not img_path or img_path in seen_paths or not Path(img_path).exists():
            continue

        try:
            pil_images.append(resize_for_vlm_answer(Image.open(img_path)))
            seen_paths.add(img_path)
            channel = doc.metadata.get("retrieval_channel", "unknown")
            labels.append(
                f"Image {len(pil_images)}: {doc.metadata.get('file_name','?')} "
                f"page {doc.metadata.get('page','?')} kind={doc.metadata.get('image_kind', 'embedded_image')} retrieved_by={channel}"
            )
        except Exception:
            continue

    return pil_images, labels

def run_vlm_text_only(messages: list) -> str:
    text_input = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_input],
        return_tensors="pt",
        padding=True,
    ).to(vlm.device)

    cleanup_cuda_memory()
    with torch.inference_mode():
        output_ids = vlm.generate(
            **inputs,
            max_new_tokens=settings.max_new_tokens,
            do_sample=settings.do_sample,
        )

    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    return vlm_processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0].strip()

def run_vlm_with_images(system_prompt: str, user_text: str, images: List[Image.Image], image_labels: List[str]) -> str:
    """Run Qwen2.5-VL with text context plus selected retrieved images."""
    content = []

    for img, label in zip(images, image_labels):
        content.append({"type": "text", "text": label})
        content.append({"type": "image", "image": img})

    content.append({"type": "text", "text": user_text})

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": content},
    ]

    # add_vision_id is useful in newer Qwen processors but not available in every version.
    try:
        text_input = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, add_vision_id=True
        )
    except TypeError:
        text_input = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    inputs = vlm_processor(
        text=[text_input],
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(vlm.device)

    cleanup_cuda_memory()
    with torch.inference_mode():
        output_ids = vlm.generate(
            **inputs,
            max_new_tokens=settings.max_new_tokens,
            do_sample=settings.do_sample,
        )

    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    return vlm_processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0].strip()

def get_candidates(question: str, faiss_stores, bm25_stores, visual_indexes=None):
    """Collect candidates from three retrieval channels.

    1. Text/caption FAISS: semantic search over text chunks and VLM image captions.
    2. BM25: keyword search for exact names, numbers, and region terms.
    3. Visual FAISS: true image-vector search over embedded images and rendered page screenshots.
    """
    regions = detect_geos_from_query(question)
    targets = regions if regions else list(faiss_stores.keys())

    seen, candidates = set(), []

    # Channel 1 + 2: existing text and caption retrieval.
    for geo in targets:
        if geo in faiss_stores:
            for doc, score in faiss_stores[geo].similarity_search_with_score(
                question, k=settings.retrieval_fetch_k
            ):
                if settings.score_threshold is None or score <= settings.score_threshold:
                    cid = doc.metadata.get("chunk_id", "")
                    if cid not in seen:
                        seen.add(cid)
                        candidates.append(
                            _clone_doc_with_metadata(
                                doc,
                                {
                                    "retrieval_channel": "text_caption_faiss",
                                    "faiss_score": float(score),
                                },
                            )
                        )

        if geo in bm25_stores:
            for doc in bm25_stores[geo].invoke(question):
                cid = doc.metadata.get("chunk_id", "")
                if cid not in seen:
                    seen.add(cid)
                    candidates.append(
                        _clone_doc_with_metadata(
                            doc,
                            {"retrieval_channel": "bm25_keyword"},
                        )
                    )

    # Channel 3: true visual search.
    # FAST MODE: only run this for visual/chart/page questions.
    # Normal summaries still retrieve image captions through text_caption_faiss, so multimodal evidence is not lost.
    if should_use_visual_retrieval(question):
        visual_indexes = visual_indexes if visual_indexes is not None else globals().get("visual_image_indexes", {})
        for doc in search_visual_image_index(question, visual_indexes, targets):
            cid = doc.metadata.get("chunk_id", doc.metadata.get("image_path", ""))
            if cid not in seen:
                seen.add(cid)
                candidates.append(doc)

    return targets, candidates

def rerank_candidates(question: str, candidates: List[Document]):
    if not candidates:
        return []

    # FAST MODE: skip cross-encoder reranking.
    # Reason: reranking reads every query-candidate pair and can be slower than retrieval itself.
    # The candidates are already ordered by retriever quality, so this is a good quick path.
    if not settings.use_reranker or reranker is None:
        return [(doc, 0.0) for doc in candidates[:settings.top_k]]

    # QUALITY MODE: cross-encoder reranking is more precise because it reads query + candidate together.
    pairs = [[question, doc.page_content[:384]] for doc in candidates]
    scores = reranker.predict(pairs).tolist()
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:settings.top_k]

def answer_question(question: str, faiss_stores, bm25_stores, visual_indexes=None) -> Dict:
    import time
    t0 = time.time()
    regions, candidates = get_candidates(question, faiss_stores, bm25_stores, visual_indexes)
    top_scored = rerank_candidates(question, candidates)

    if not top_scored:
        return {
            "question": question,
            "answer": "I do not have the answer in the provided documents.",
            "geo_used": regions,
            "mode": "no_match",
            "sources": [],
        }

    top_docs = [doc for doc, _ in top_scored]
    geo_label = ", ".join(regions) if regions else "ALL"

    # Build text context. For image docs, this includes VLM semantic captions.
    text_context = format_text_context(top_docs)

    user_text = (
        f"Region: {geo_label}\n\n"
        f"Context:\n{text_context}\n\n"
        f"Question: {question}\n\n"
        "Answer with specific source-grounded details. "
        "When images are provided, use them only if they support the retrieved context."
    )

    # Only send actual images/pages to Qwen when the question needs visual reasoning.
    # Reason: direct image input is expensive and can cause CUDA OOM on 15GB Colab GPUs.
    answer_images, image_labels = ([], [])
    if should_use_direct_images(question):
        answer_images, image_labels = load_answer_images(top_docs)

    if answer_images:
        mode = "vision_direct"
        try:
            answer = run_vlm_with_images(SYSTEM_PROMPT, user_text, answer_images, image_labels)
        except torch.cuda.OutOfMemoryError:
            # Safe fallback: clear GPU cache and retry without images.
            cleanup_cuda_memory()
            mode = "text_fallback_after_oom"
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
            ]
            answer = run_vlm_text_only(messages)
    else:
        mode = "text"
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
        ]
        answer = run_vlm_text_only(messages)

    sources = [
        {
            "type": d.metadata.get("element_type"),
            "file": d.metadata.get("file_name"),
            "page": d.metadata.get("page"),
            "score": round(float(s), 3),
            "retrieval_channel": d.metadata.get("retrieval_channel"),
            "visual_score": (
                round(float(d.metadata["visual_score"]), 3)
                if d.metadata.get("visual_score") is not None
                else None
            ),
            "has_image": bool(d.metadata.get("image_path")),
            "image_path": d.metadata.get("image_path"),
            "image_caption_enriched": bool(d.metadata.get("image_caption_enriched", False)),
        }
        for d, s in top_scored
    ]

    return {
        "question": question,
        "answer": answer,
        "geo_used": regions,
        "mode": mode,
        "latency_seconds": round(time.time() - t0, 2),
        "fast_mode": settings.fast_mode,
        "use_reranker": settings.use_reranker,
        "used_visual_retrieval": should_use_visual_retrieval(question),
        "sources": sources,
    }

print("✓ Retrieval + direct vision answer pipeline ready — FAST MODE enabled")
print("  Channels: text_caption_faiss + bm25_keyword + visual_embedding_page_or_image")


✓ Retrieval + direct vision answer pipeline ready — FAST MODE enabled
  Channels: text_caption_faiss + bm25_keyword + visual_embedding_page_or_image


---
## Step 14 — Test queries

In [18]:
def print_result(result: Dict) -> None:
    print(f"Question: {result['question']}")
    print(f"Mode    : {result['mode'].upper()}")
    print(f"\nAnswer:\n{result['answer']}")
    print()
    if result['sources']:
        print("Sources:")
        for s in result['sources']:
            img = " [img]" if s["has_image"] else ""
            enriched = " [caption-indexed]" if s.get("image_caption_enriched") else ""
            visual = f" visual={s['visual_score']:+.2f}" if s.get("visual_score") is not None else ""
            channel = s.get("retrieval_channel") or "unknown"
            print(
                f"  {s['type']:10} {s['file']} p.{s['page']} "
                f"rerank={s['score']:+.2f} channel={channel}{visual}{img}{enriched}"
            )
    print()

# Query 1
result = answer_question(
    "What is the refund policy for APAC?",
    geo_vectorstores, geo_bm25_stores,
    visual_image_indexes,
)
print_result(result)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Question: What is the refund policy for APAC?
Mode    : TEXT

Answer:
The provided context does not contain any information about refund policies for different regions such as APAC, EMEA, or AMER. There is no mention of refund policies in the given text or visual evidence. Therefore, based on the information provided, I do not have the answer to the refund policy for APAC.

Sources:
  text       invalid-pdf-structure-pdfminer-one-page.pdf p.1 rerank=+0.00 channel=text_caption_faiss
  vlm_page_image embedded-link.pdf p.0 rerank=+0.00 channel=text_caption_faiss [img] [caption-indexed]



In [19]:
# Query 2 — with images
result = answer_question(
    "Describe charts and diagrams in the documents of APAC",
    geo_vectorstores, geo_bm25_stores,
    visual_image_indexes,
)
print_result(result)

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Question: Describe charts and diagrams in the documents of APAC
Mode    : VISION_DIRECT

Answer:
The documents provide information about charts and diagrams within the context of APAC, EMEA, and AMER regions. Specifically, the document discusses a network diagram that represents connections between various entities, suggesting a flow or interaction among them. This network is related to cloud computing, with nodes representing different services or data points.

The network diagram includes elements like numbers and labels such as "Cloud," "Speaker," "Document," and "House." These labels represent different categories or entities within the network. The relationships depicted suggest a flow or interaction among these entities, indicating a collaborative or interconnected system.

The second diagram mentioned is a simple circular diagram with concent

Sources:
  vlm_image  invalid-pdf-structure-pdfminer-one-page.pdf p.0 rerank=+0.00 channel=text_caption_faiss [img] [caption-indexed]
  v

In [20]:
# Query 3
result = answer_question(
   "Summary of all the the documents of APAC documents",
    geo_vectorstores, geo_bm25_stores,
    visual_image_indexes,
)
print_result(result)

Question: Summary of all the the documents of APAC documents
Mode    : TEXT

Answer:
The documents provide information about the National Cybersecurity Strategy launched by the U.S. Government Accountability Office (GAO) in June 2023 for the Asia-Pacific region (APAC). The strategy aims to defend critical infrastructure, disrupt and dismantle cyber threats, shape market forces to drive security and resilience, invest in a resilient future, and forge international partnerships.

The strategy is structured around five pillars, as highlighted in Figure 1:

1. **Defend**: This pillar focuses on defending critical infrastructure.
2. **Shape Market Forces**: This pillar aims to shape market forces to drive security and resilience.
3. **Invest in a

Sources:
  vlm_page_image invalid-pdf-structure-pdfminer-one-page.pdf p.1 rerank=+0.00 channel=text_caption_faiss [img] [caption-indexed]
  vlm_page_image invalid-pdf-structure-pdfminer-one-page.pdf p.0 rerank=+0.00 channel=text_caption_faiss [img

In [21]:
# Query 3
result = answer_question(
   "take any image and explain what it represent",
    geo_vectorstores, geo_bm25_stores,
    visual_image_indexes,
)
print_result(result)

Question: take any image and explain what it represent
Mode    : VISION_DIRECT

Answer:
The image represents a simple circular diagram consisting of two concentric circles. The outer circle is larger and black, while the inner circle is smaller and white. There are no labels, examples, or additional elements to indicate relationships or comparisons. This type of diagram can be used in educational materials or presentations to illustrate fundamental geometric concepts related to circles and their properties.

Sources:
  vlm_image  invalid-pdf-structure-pdfminer-one-page.pdf p.1 rerank=+0.00 channel=text_caption_faiss [img] [caption-indexed]
  vlm_image  invalid-pdf-structure-pdfminer-one-page.pdf p.1 rerank=+0.00 channel=text_caption_faiss [img] [caption-indexed]



---
## What changed in this upgraded version

| Upgrade | Why it matters |
|---|---|
| Fixed execution order | `settings` is now created before any cell tries to use it. |
| Persistent Drive storage | Data, indexes, extracted images, rendered page screenshots, and caches survive Colab runtime restarts. |
| Simple image preprocessing | Converts visuals to RGB and only downscales very large images; model processors handle resizing/normalization. |
| Full-page screenshot indexing | Catches vector charts, tables, diagrams, and layout that embedded-image extraction may miss. |
| Qwen2.5-VL import update | Uses the current `Qwen2_5_VLForConditionalGeneration` import path. |
| VLM caption visual indexing | Embedded images and page screenshots become searchable through captions containing visible text, labels, numbers, and relationships. |
| True visual embedding indexing | Embedded images and page screenshots are also embedded as pixels with SigLIP2 and stored in a separate FAISS visual index. |
| Three retrieval channels | The system combines text/caption FAISS, BM25 keyword search, and visual-vector search. |
| Direct VLM visual answering | Retrieved embedded images or page screenshots can be passed directly into Qwen2.5-VL for final reasoning. |
| Metadata image paths | FAISS stores vectors/doc IDs; the image files stay on disk, and `image_path` points back to them. |
| Comments and reasons | Important cells explain what is happening and why that technique is used. |

### Retrieval type in this notebook

This version has three levels of retrieval:

```text
Level 1:
text/table/json → BGE text embedding → FAISS/BM25 retrieval

Level 2:
embedded image or page screenshot → Qwen2.5-VL caption → BGE text embedding → FAISS/BM25 retrieval

Level 3:
embedded image or page screenshot pixels → SigLIP2 visual embedding → FAISS visual retrieval
```

So retrieval is not only caption-based. It includes a **true visual-vector retrieval path**, and it can also retrieve full-page screenshots.

### Why page screenshots matter

PDF image extraction is not enough by itself. Many PDFs draw charts and tables as vector graphics, which means `page.get_images()` may find nothing useful. Rendering each page as an image lets the system index the full page layout, including vector charts, tables, legends, diagrams, and nearby explanatory text.

### Folder structure expected

```text
Google Drive/RAG_data/data/APAC/
Google Drive/RAG_data/data/EMEA/
Google Drive/RAG_data/data/AMER/
```

Upload PDFs to the right folder, then run the notebook from top to bottom.
